# Earlier findings checked on dandiset 001538 version draft

This notebook belongs to the reconciliation of attempt zhai-2025-02 with attempt zhai-2025-01
(docs/protocol.md, "Repeat attempts"). zhai-2025-01 was made on version 0.260527.1302 and
zhai-2025-02 on the draft. Of the fifteen findings of zhai-2025-01, findings 3, 4, 9, 10, 11, 12,
14 and 15 are not reported by zhai-2025-02, and each is checked here directly on
[DANDI:001538](https://dandiarchive.org/dandiset/001538/draft), version draft. Findings 1, 6 and 7
are reported by zhai-2025-02 in part; the parts it does not report are checked here too. Each
section opens with the earlier statement. The files are streamed with the entry's helper module,
`lid.py`, and only the tables, headers and stimulus series the checks need are read. Finding 14 is
about the paper and is checked on the paper's per-cell tables (doi:10.5281/zenodo.17307059).
The conclusion for each finding is in the reconciliation block of `attempts/zhai-2025-02.yaml`.

In [ ]:
import re
from concurrent.futures import ProcessPoolExecutor
import warnings
warnings.filterwarnings("ignore")
import h5py
import numpy as np
import pandas as pd
from scipy import stats
import lid

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)
info = lid.version_info()
assets = lid.list_assets()
print("dandiset", lid.DANDISET, "version", lid.VERSION, "| modified", info["modified"], "| NWB files", len(assets))

In [ ]:
def _s(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def header(a):
    """Header fields of one file, and every object that carries a description."""
    f = lid.open_asset(a.asset_id)
    get = lambda k: _s(f[k][()]) if k in f else ""
    names, descs = [], []
    def visit(n, o):
        names.append(n)
        if "description" in o.attrs:
            descs.append((n, _s(o.attrs["description"])))
    f.visititems(visit)
    out = dict(path=a.path, kind=a.kind, name_genotype=a.genotype,
               related=[_s(x) for x in f["general/related_publications"][:]] if "general/related_publications" in f else [],
               session_description=get("session_description"), experiment_description=get("general/experiment_description"),
               subject_genotype=get("general/subject/genotype"), names=names, descs=descs)
    f.close()
    return out


def patch_tables(a):
    """The per-sweep tables of one patch-clamp file, its dendritic stimulus series and, for the
    Sr2+-oEPSC files, the light pulses found in each sweep."""
    f = lid.open_asset(a.asset_id)
    out = dict(path=a.path, date=a.date, cell_type=a.cell_type, name_drug=a.drug,
               description=_s(f["session_description"][()]))
    irt = f["general/intracellular_ephys/intracellular_recordings"]
    resp, stim, cell = irt["responses/response"][:], irt["stimuli/stimulus"][:], irt["cell_id"][:]
    if "processing/icephys/RecordingsIndexTable" in f:
        t = f["processing/icephys/RecordingsIndexTable"]
        rit = pd.DataFrame({k: lid._str(t[k][:]) if t[k].dtype.kind == "O" else t[k][:]
                            for k in ["cell_id", "condition", "dendrite_type", "site", "stimulus_current_pA", "irt_row"]})
        rit["drug"] = lid._str(irt["drug"][:])[rit.irt_row]
        lengths = {}
        for col in ["line_scan_alexa568", "line_scan_fluo4"]:
            ok = []
            for start, count, ref in (t[col][:] if col in t else [(-1, 0, None)] * len(rit)):
                if start < 0:
                    ok.append(False)
                    continue
                ds = f[ref]
                lengths.setdefault(ds.name, ds["data"].shape[0])
                ok.append(start + count <= lengths[ds.name])
            rit[col + "_ok"] = ok
        out["rit"] = rit
        out["dendritic_stimulus"] = [
            dict(series=n, samples=len(d := f["stimulus/presentation"][n]["data"][:]),
                 peak=float(np.max(np.abs(d))) if len(d) else np.nan,
                 unit=_s(f["stimulus/presentation"][n]["data"].attrs.get("unit", "")),
                 conversion=float(f["stimulus/presentation"][n]["data"].attrs.get("conversion", 1.0)))
            for n in f["stimulus/presentation"] if "Dendritic" in n]
    else:
        p = f["intervals/optogenetic_pulses_table"]
        ps, pc = p["start_time"][:], p["cell_id"][:]
        ts, rows = {}, []
        for i in range(len(resp)):
            start, count, ref = resp[i]
            ds = f[ref]
            ts.setdefault(ds.name, lid._timestamps(ds))
            sweep = ts[ds.name][start:start + count]
            mine = ps[(pc == cell[i]) & (ps >= sweep[0]) & (ps <= sweep[-1])]
            rows.append(dict(cell=int(cell[i]), pulses=len(mine), pulse_s=float(mine[0] - sweep[0]) if len(mine) else np.nan,
                             stimulus_rows=int(stim[i][1]) if stim[i][0] >= 0 else 0))
        out["sr"] = pd.DataFrame(rows)
        types = set()
        f.visititems(lambda n, o: types.add(_s(o.attrs.get("neurodata_type", ""))))
        out["types"] = sorted(types - {""})
    f.close()
    return out


with ProcessPoolExecutor(16) as ex:
    heads = pd.DataFrame(list(ex.map(header, [a for _, a in assets.iterrows()])))
    patch = assets[assets.kind.isin(["icephys", "icephys+ophys"]) & assets.path.str.contains("SPN")]
    ptabs = list(ex.map(patch_tables, [a for _, a in patch.iterrows()]))
cc = [p for p in ptabs if "rit" in p]
srt = [p for p in ptabs if "sr" in p]
R = pd.concat([p["rit"].assign(path=p["path"], date=p["date"], cell_type=p["cell_type"], name_drug=p["name_drug"]) for p in cc],
              ignore_index=True)
tables = lid.paper_tables()
print("headers read:", len(heads), "| current-clamp files:", len(cc), "| Sr2+-oEPSC files:", len(srt))

## Earlier finding 1 (data_missing), the part not reported again

> The dandiset holds no patch-clamp recordings from unlesioned control or 6-OHDA-lesioned mice,
> which make up the control and 6-OHDA groups of Figures 1E, 1F, 1I, 3C, 3D and 3F (10, 12, 11 and
> 9 dSPNs; 21, 11, 14 and 9 iSPNs), nor the oxo-M line scans of Figure 7E or the open-field data of
> Figure 10. The paper's raw data on Zenodo (doi:10.5281/zenodo.17260198) list folders for all of
> these.

zhai-2025-02 reports the control and 6-OHDA recordings and the open-field data again (its finding
11). The oxo-M line scans are checked here: the files by session-name fields and kind, and the
oxo-M cells by condition.

In [ ]:
print(assets.groupby(["cell_type", "state", "drug", "kind"]).size().to_string())
ox = R[R.condition.str.contains("naive")]
print("\noxo-M files: cells by condition and compartment")
print(ox.groupby(["condition", "dendrite_type"]).apply(lambda g: g.groupby(["date", "cell_id"]).ngroups).to_string())
print("\npatch-clamp files of unlesioned or 6-OHDA mice:",
      int(patch.state.isin(["UnlesionedControl", "LesionedControl"]).sum()),
      "| files of those states by kind:", assets[assets.state.isin(["UnlesionedControl", "LesionedControl"])].kind.replace("", "spine stacks").value_counts().to_dict())

## Earlier finding 3 (metadata_error)

> Session names state a drug (D1RaSch, D2RaSul) for files that also hold baseline on-state
> sweeps. 11 cells in such files have a baseline condition; 7 of them were also recorded with the
> drug and 4 are baseline only (dSPN 20190507 cell 1, iSPN 20190408 cell 1, 20190409 cell 1 and
> 20190508 cell 2). The condition of each sweep is only in the RecordingsIndexTable column
> condition_subfolder; grouping by session name puts baseline cells in the drug groups.

The somatic cells of the files whose session name states a drug, with the conditions their
sweeps carry, and the columns that give each sweep's condition and drug.

In [ ]:
named = R[(R.dendrite_type == "Soma") & R.name_drug.isin(["D1RaSch", "D2RaSul"])]
c3 = named.groupby(["name_drug", "cell_type", "date", "cell_id"]).condition.apply(lambda x: sorted(set(x))).reset_index()
c3["baseline"] = c3.condition.apply(lambda x: any("with" not in y for y in x))
c3["with_drug"] = c3.condition.apply(lambda x: any("with" in y for y in x))
print(c3.to_string(index=False))
print("\ncells with a baseline condition:", int(c3.baseline.sum()), "| also recorded with the drug:",
      int((c3.baseline & c3.with_drug).sum()), "| baseline only:", int((c3.baseline & ~c3.with_drug).sum()))
print(c3[c3.baseline & ~c3.with_drug][["cell_type", "date", "cell_id"]].to_string(index=False))
print("\ncondition and drug columns of the intracellular recordings table, in these files:")
print(named.groupby(["name_drug", "condition", "drug"]).size().to_string())

## Earlier finding 4 (metadata_error)

> The genotypes of the CDGI behaviour mice contradict each other. Subject genotype is Wild-type
> CDGI and every session name says WT for all of them; the AIM table's genotype column says CDGI
> Knockout, Wild Type or unknown, changing between sessions of one animal; the video files of mice
> 1945, 2519, 2587, 2828 and 6572 say CDGI KO. The paper lists 2587 as wild type and the others as
> KO.

For every behaviour file (AIM scores and videos): the mouse and group in the session description,
the genotype in the session name and the subject genotype, per mouse, against the groups of the
paper's Figure 8 table.

In [ ]:
beh = heads[heads.path.str.contains("ses-pan") & (heads.kind != "ophys")].copy()
m = beh.session_description.str.extract(r"of mouse (\d+) \(([^,]+), Figure (\d+)\)")
beh["mouse"], beh["group"], beh["figure"] = m[0].astype(int), m[1], m[2].astype(int)
beh["subject_genotype"] = beh.subject_genotype.str.split(";").str[0]
beh["file"] = np.where(beh.kind == "image", "video", "AIM")
per = beh.groupby(["figure", "mouse"]).agg(groups=("group", lambda x: sorted(set(x))), name_genotype=("name_genotype", lambda x: sorted(set(x))),
                                           subject_genotype=("subject_genotype", lambda x: sorted(set(x))), files=("file", lambda x: sorted(set(x)))).reset_index()
mixed = per[(per.groups.str.len() > 1) | (per.name_genotype.str.len() > 1) | (per.subject_genotype.str.len() > 1)]
print("behaviour files:", len(beh), "| mice:", len(per), "| mice with more than one label:", len(mixed))
print(beh.groupby(["figure", "group", "name_genotype", "subject_genotype"]).mouse.nunique().to_string())
p8 = tables["Fig 8"]["Fig 8B_total AIMs"].set_axis(["mouse", "pgroup", "session", "total"], axis=1)
p8["mouse"] = p8.mouse.astype(str).str.extract(r"(\d+)")[0].astype(int)
pg = p8.groupby("mouse").pgroup.apply(lambda x: sorted(set(x))).rename("paper_group")
cmp8 = per[per.figure == 8].merge(pg, left_on="mouse", right_index=True, how="left")
print("\nmice 1945, 2519, 2587, 2828 and 6572:")
print(cmp8[cmp8.mouse.isin([1945, 2519, 2587, 2828, 6572])].to_string(index=False))
both = cmp8.dropna(subset=["paper_group"])
agree = both.apply(lambda r: ("knockout" in r.groups[0]) == any("KO" in g for g in r.paper_group), axis=1)
print("\nFigure 8 mice in the paper's table:", len(both), "| dandiset group agrees with the paper's:", int(agree.sum()))

## Earlier finding 6 (metadata_error), the parts not reported again

> Three dendritic site series hold six trials, two sites under one name, while the cell has no
> series of the other compartment with that number (iSPN 20160526 cell 3 Distal1, iSPN 20160608
> cell 1 Proximal1, dSPN 20170204 cell 1 Proximal1); their first and last three trials differ in
> baseline and area. Split, they give the paper's values for these cells. One series holds a single
> trial (iSPN 20160524 cell 2 Proximal1), one two and one four.

zhai-2025-02 reports the first two cells (its finding 1). The sites of all four named cells, and
every site whose trial count is not three.

In [ ]:
den = R[R.dendrite_type != "Soma"]
sites = den.groupby(["cell_type", "date", "cell_id", "condition", "dendrite_type", "site"]).size().rename("trials").reset_index()
for ct, d, c in [("iSPN", "20160526", 3), ("iSPN", "20160608", 1), ("dSPN", "20170204", 1), ("iSPN", "20160524", 2)]:
    print(sites[(sites.cell_type == ct) & (sites.date == d) & (sites.cell_id == c)].to_string(index=False), "\n")
print("sites by number of trials:", sites.trials.value_counts().sort_index().to_dict())
print(sites[sites.trials != 3].to_string(index=False))

## Earlier finding 7 (data_missing), the part not reported again

> Two dendritic cells lack the sites the paper's values need. dSPN on+SCH 20190411 cell 1 has
> only its second pair of sites, which gives 0.584 against the paper's 0.228; [...] iSPN on+sul
> 20190508 cell 3 has only a distal site; with the paper's 0.217 the sulpiride test gives the
> paper's P (0.0545), without it P = 0.0097. [...]

zhai-2025-02 reports the first cell again (its finding 2). The sites of iSPN 20190508 cell 3, and
its index from the entry's dendritic rule, against the paper's 0.217.

In [ ]:
print(sites[(sites.cell_type == "iSPN") & (sites.date == "20190508")].to_string(index=False))
a = patch[(patch.cell_type == "iSPN") & (patch.date == "20190508")].iloc[0]
_, d7 = lid.load_patch_file(a)
cells7, pairs7, _ = lid.dendritic_index(d7)
print(cells7[["date", "cell", "condition", "index"]].round(3).to_string(index=False))

## Earlier finding 9 (metadata_error)

> The stimulus series of every dendritic sweep (CurrentClampSeries_CellN_Dendritic_Stimulus) is all
> zero, though the three spikes per sweep show the 2 nA injections were delivered; and the index
> table gives no kymograph slice for 53 of 997 dendritic sweeps.

The peak of each dendritic stimulus series, and whether each dendritic sweep's two line-scan
(kymograph) references point at data.

In [ ]:
ds = pd.DataFrame([dict(path=p["path"], **s) for p in cc for s in p["dendritic_stimulus"]])
print("dendritic stimulus series:", len(ds), "| all zero:", int((ds.peak == 0).sum()),
      "| peak values:", ds.peak.value_counts().to_dict(), "| units:", ds.unit.value_counts().to_dict(),
      "| conversion:", ds.conversion.value_counts().to_dict(), "| peak (nA):", (ds.peak * ds.conversion * 1e9).round(3).value_counts().to_dict())
ok = den.line_scan_alexa568_ok & den.line_scan_fluo4_ok
print("dendritic sweeps:", len(den), "| without a valid kymograph slice:", int((~ok).sum()))

## Earlier finding 10 (metadata_error)

> In the Sr2+-oEPSC files the light-pulse OptogeneticSeries is all zero in 69 of 140 sweeps, and
> the intracellular recordings table references no stimulus (its stimulus rows are empty). The
> pulse time (0.02 s) is only in the optogenetic epochs table.

For each Sr2+-oEPSC sweep: the light pulses in the file's optogenetic pulses table that fall in
the sweep, their time from the sweep start, and the length of the stimulus the intracellular
recordings table references; and the neurodata types in the files.

In [ ]:
sr = pd.concat([p["sr"].assign(path=p["path"]) for p in srt], ignore_index=True)
print("files:", len(srt), "| cells:", sr.groupby(["path", "cell"]).ngroups, "| sweeps:", len(sr))
print("light pulses per sweep:", sr.pulses.value_counts().to_dict(), "| pulse time from sweep start (s):", sr.pulse_s.round(4).value_counts().to_dict())
print("sweeps whose stimulus reference is empty:", int((sr.stimulus_rows == 0).sum()))
types = sorted(set(t for p in srt for t in p["types"]))
print("OptogeneticSeries in any file:", "OptogeneticSeries" in types)
print("optogenetic and stimulus types:", [t for t in types if "Optogenetic" in t or "Stimulus" in t])

## Earlier finding 11 (metadata_error)

> The GRAB-ACh background reference images are described as taken at zero laser power, but their
> mean (381 to 525) is above the signal (F0 179 to 355), so they cannot be the PMT background the
> Methods subtract.

Every object of the GRAB-ACh3.0 files whose name or description mentions a background, the PMT or
the laser power, and the description of the reference images the files hold.

In [ ]:
grab = heads[heads.kind == "ophys"]
pat = re.compile(r"background|laser power|PMT|zero", re.I)
hits = [(h.path.split("/")[0], n) for _, h in grab.iterrows() for n in h.names if pat.search(n)]
hits += [(h.path.split("/")[0], n, d[:200]) for _, h in grab.iterrows() for n, d in h.descs if pat.search(d)]
print("GRAB-ACh3.0 files:", len(grab), "| objects that mention a background, the PMT, the laser power or zero:", len(hits))
for x in hits:
    print(x)
refs = sorted({d for _, h in grab.iterrows() for n, d in h.descs if n.endswith("ImagesPrairieViewReferences")})
print("\nreference image descriptions:", refs)

## Earlier finding 12 (metadata_error)

> The files cite the bioRxiv preprint (related_publications) and describe the behaviour data with
> the preprint's figure numbers ("Figure 7J" for the CDGI AIM scores, "Figure 8" for the M1R
> study), which are Figures 8 and 10 in the published paper the dandiset names.

The related publications of every file, any mention of the preprint, and the figures the
behaviour files name.

In [ ]:
print("related_publications over", len(heads), "files:", heads.related.astype(str).value_counts().to_dict())
pre = re.compile(r"biorxiv|preprint|10\.1101|7J", re.I)
texts = [(h.path, t) for _, h in heads.iterrows() for t in [h.session_description, h.experiment_description] + [d for _, d in h.descs]]
print("descriptions that mention the preprint or Figure 7J:", sum(bool(pre.search(t)) for _, t in texts))
print("behaviour files by figure named and group:", beh.groupby(["figure", "group"]).size().to_dict())

## Earlier finding 14 (paper_inconsistency)

> Recomputed on the paper's own per-cell tables, some reported P values differ; the oxo-M test in
> CDGI KO gives P = 0.47 where the paper reports 0.910, and the rotation test of Figure 8A gives
> 0.012 where the text reports 0.0092. Others agree closely (0.009766 against 0.0098 for oxo-M in
> wild type).

On the paper's tables: the two-sided Wilcoxon tests of Figure 7E, before against after oxo-M,
paired by experiment number, and the two-sided Mann-Whitney test of Figure 8A, as the claims file
states them.

In [ ]:
e7 = tables["Fig 7"]["Fig 7E_oxoM effect on DE"].set_axis(["id", "genotype", "treatment", "v"], axis=1)
for gt in ["WT", "KO"]:
    w = e7[e7.genotype == gt].pivot_table(index="id", columns="treatment", values="v")
    r = stats.wilcoxon(w["before (control)"], w["oxoM"])
    print(f"Figure 7E {gt}: n={len(w)}, Wilcoxon W={r.statistic:.1f}, P={r.pvalue:.4g}")
a8 = tables["Fig 8"]["Fig 8A_rotations 5th"].set_axis(["id", "group", "v"], axis=1)
r = stats.mannwhitneyu(a8[a8.group == "KO"].v, a8[a8.group == "WT"].v, alternative="two-sided")
print(f"Figure 8A: KO n={int((a8.group == 'KO').sum())}, WT n={int((a8.group == 'WT').sum())}, Mann-Whitney U={r.statistic:.1f}, P={r.pvalue:.4g}")

## Earlier finding 15 (other, about the data)

> 18 somatic cells miss one or more current steps between 20 and 300 pA and 14 repeat a step;
> spike counts for these cells cannot be complete.

For each somatic cell and condition, the steps from 20 to 300 pA (in steps of 20 pA) that have no
sweep and the steps recorded more than once, from the RecordingsIndexTable.

In [ ]:
steps = list(range(20, 301, 20))
rows = []
for (path, cell, cond), g in R[R.dendrite_type == "Soma"].groupby(["path", "cell_id", "condition"]):
    c = g.stimulus_current_pA.round().astype(int)
    rows.append(dict(file=re.search(r"ses-(.*)\.nwb", path).group(1), cell=cell, condition=cond,
                     missing=[s for s in steps if s not in set(c)], repeated=sorted(set(c[c.duplicated()]))))
f15 = pd.DataFrame(rows)
print("somatic cells:", len(f15), "| missing a step from 20 to 300 pA:", int((f15.missing.str.len() > 0).sum()),
      "| with a repeated step:", int((f15.repeated.str.len() > 0).sum()),
      "| both:", int(((f15.missing.str.len() > 0) & (f15.repeated.str.len() > 0)).sum()))
print(f15[(f15.missing.str.len() > 0) | (f15.repeated.str.len() > 0)].to_string(index=False))